---
title: "2.3 Creating and transforming variables"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/03-transforming.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/03-transforming.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 35 minutes, including exercises · **Before you begin:** Complete [2.2 Selecting rows and columns](../02-data/02-selecting.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Create columns with arithmetic and clear units.
- Create a yes/no indicator and update selected cells with .loc.
- Use a natural logarithm while checking positive inputs.

## Why it matters

Researchers often need a variable that the raw file does not contain: GDP in thousands, a threshold indicator, or log GDP per person. A **transformation** applies a rule to an existing measurement. It changes how a quantity is expressed; it does not add evidence. We first use made-up numbers for fictional countries, then apply the same explicit rules to the running panel.

## Worked example

### Change a unit

A formula involving a pandas column acts on each row. Divide dollars by 1000 to create a new column measured in thousands of dollars. Assignment to a new column name adds it to the table.

In [ ]:
import pandas as pd
small = pd.DataFrame({"country": ["Northland", "Eastria"], "gdp_pc": [15000.0, 30000.0], "life_exp": [74.0, 78.0]})
small["gdp_pc_thousands"] = small["gdp_pc"] / 1000
small

The new values are 15.0 and 30.0 thousand dollars per person. The original dollar column remains available. Names that mention the new unit help prevent mistakes in tables and regression interpretations.

### Create a yes/no indicator

An **indicator** records whether a condition holds. A comparison returns Booleans. `.astype(int)` converts True to 1 and False to 0 here, where every condition has a known value. The threshold is an illustrative teaching choice.

In [ ]:
small["life_exp_at_least_75"] = (small["life_exp"] >= 75).astype(int)
small[["country", "life_exp", "life_exp_at_least_75"]]

Northland receives 0 and Eastria 1. Do not treat an unknown original measurement as a known negative answer; the next lesson introduces missing values and nullable types.

### Change selected cells in one step

`.loc` can assign to a named column for the rows meeting a condition. Use a practice copy and explicitly label its two groups.

In [ ]:
practice = small.copy()
practice["illustrative_group"] = "Below 75"
practice.loc[practice["life_exp"] >= 75, "illustrative_group"] = "75 or more"
practice[["country", "illustrative_group"]]

The labels match the indicator. Use a single `.loc[...] = ...` assignment rather than selecting a temporary column and then editing it. The separate practice copy makes it clear which table is being changed.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Express total production in billions

Load a fresh copy of the panel. GDP per person multiplied by population gives an implied total in the source's inflation-adjusted dollars. Divide by one billion, written as `1_000_000_000`, for a convenient display unit.

In [ ]:
countries = pd.read_csv(data_file)
countries["gdp_billions"] = countries["gdp_pc"] * countries["population"] / 1_000_000_000
countries.loc[(countries["country"] == "Canada") & (countries["year"] == 2007), ["country", "year", "gdp_pc", "population", "gdp_billions"]]

The implied total for Canada is about 1,212.7 billion of the extract's inflation-adjusted dollars. It is derived from these two source columns, not an independently retrieved national-accounts series. Do not compare it directly with current-dollar GDP from another source.

### Take a natural logarithm

A **natural logarithm** expresses multiplication as addition: doubling a level always adds the same amount to its log. This can help compare countries whose GDP levels differ greatly. NumPy is a library for numerical calculations; its `np.log` function works on each number in a column. First verify that all GDP-per-person values are positive.

In [ ]:
import numpy as np
print((countries["gdp_pc"] > 0).all())
countries["log_gdp_pc"] = np.log(countries["gdp_pc"])
print(round(float(np.log(2)), 3))
countries[["country", "year", "gdp_pc", "log_gdp_pc"]].head()

The check is True. A doubling adds about 0.693 to the natural log. The transformed column is not measured in dollars and a one-unit log increase is not a one-dollar increase. Logging GDP does not make an association causal. Zero or negative values require an explicit analytic decision; do not silently add 1 to avoid the problem.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Create a population unit

Add `population_millions`, population divided by one million. Display it for Canada in 2007.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
countries["population_millions"] = countries["population"] / 1_000_000
canada_2007 = countries.loc[(countries["country"] == "Canada") & (countries["year"] == 2007)]
print(canada_2007["population_millions"].iloc[0])

The value is 33.390141 million people. The original population is still stored in people.

</details>

### 2. Create an illustrative indicator

For the 2007 cross-section, create `at_least_80` equal to 1 when life expectancy is at least 80 years and 0 otherwise. Show Canada's result.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
year_2007 = countries.loc[countries["year"] == 2007].copy()
year_2007["at_least_80"] = (year_2007["life_exp"] >= 80).astype(int)
year_2007.loc[year_2007["country"] == "Canada", ["country", "life_exp", "at_least_80"]]

Canada receives 1 for its 80.653-year observation. The indicator is a documented threshold decision, not an official classification of a country or person.

</details>

### 3. Check a log difference

Using made-up levels 10000 and 20000, subtract their natural logarithms and compare with the log of 2.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
doubling_gap = np.log(20000) - np.log(10000)
print(round(float(doubling_gap), 3))
print(round(float(np.log(2)), 3))

Both displayed values are 0.693. Equal ratios produce equal log differences, even when the dollar difference changes.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A required column is absent

A formula is only meaningful for the columns actually present.

In [ ]:
try:
    countries["total_gdp"] / countries["population"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

The source contains gdp_pc, not total_gdp. Inspect its data dictionary before trying to recreate a measurement that is already per person.

### A logarithm has an invalid domain

A function's domain is the set of inputs for which it is defined. For a real-valued natural log, inputs must be strictly positive. The scalar math function raises an explicit error for zero.

In [ ]:
try:
    import math
    math.log(0)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

The message says “math domain error.” NumPy may instead warn and produce an infinite or missing value, so the positive-input check is essential before transforming a column.

## Recap

- Name transformed variables and their units clearly.
- A column formula applies row by row.
- Use .loc to change selected rows in a single assignment.
- Check the domain before logging and explain ratios in plain language.

**Next:** [2.4 Missing values and data types](../02-data/04-missing-types.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
Creating a pandas column is similar to Stata's generate, an R transformation, or a spreadsheet formula filled down a column. With pandas, keep the whole column's units and missing-value behaviour in mind.
:::